# Leniwy odczyt, cache i ładowanie równoległe

Jak ograniczyć pamięć i przygotowywać wybrane pliki równolegle bez wczytywania całego zbioru? Notebook korzysta z lokalnych plików przez bibliotekę `aminor_msa`; nie uruchamia pobierania ani nie rozpakowuje całego ZIP-a. Każdy notebook można uruchomić osobno przez **Restart Kernel → Run All**.

## Introduction

Zmień wybór w `settings.toml` albo w jawnie nazwanych zmiennych przy wywołaniu. Kontrolki działają w uruchomionym kernelu; po zmianie rodziny w kontrolce uruchom ponownie komórki, które odczytują `browser.selected_family`.

## Assumptions

- Środowisko: `uv sync --extra notebook`; w edytorze wybierz `.venv/bin/python`.
- Dane są już w `data/`. Jeśli ich brakuje, użyj z katalogu projektu `uv run python -m assets.scripts.download`; szczegóły w [acquisition.md](../../../docs/acquisition.md).
- To kontrola formatów i jawnych kluczy źródłowych. Nie powstaje mapowanie reszt struktury do kolumn MSA ani zbiór treningowy.
- Przykład wybiera Antoninę Olejnik, rodzinę Fluoride `RF01734` i strukturę `3vrs`. Inne rodziny mogą nie mieć struktur lub raportów DSSR.

## Notation

`workers` ogranicza liczbę workerów; `prefetch` liczbę wysłanych zadań przed konsumpcją. `cache_entries` liczy obiekty, nie bajty. `SourceRequest(kind, accession, variant)` jest jednoznacznym kluczem artefaktu.

[Reader: odczyt, klucze i przygotowanie do treningu](../../../docs/reader.md).

In [ ]:
from pathlib import Path
from IPython.display import display
import pandas as pd

from aminor_msa import DataStore
from aminor_msa.utils.config import read_toml
from aminor_msa.utils.paths import find_project_root
from aminor_msa.visualization import ViewStyle

ROOT = find_project_root()
NOTEBOOK_DIR = ROOT / "assets/analysis/base_functionality/26_10_06"
SETTINGS_PATH = NOTEBOOK_DIR / "settings.toml"
settings = read_toml(SETTINGS_PATH)
selection = settings["selection"]
limits = settings["display"]
style = ViewStyle(**{key: limits[key] for key in ("max_sequences", "max_columns", "font_size", "width", "height")})
store = DataStore(ROOT / "data")
display({"data_root": str(store.root), "selection": selection, "initial_state": store.load_state()})

## Observe lazy state and cache

### Methodology

**Theoretical methodology.** Utworzenie sklepu i uchwytu nie odczytuje plików. Małe indeksy powstają na żądanie. Domyślnie odczyt źródła nie pozostawia go w cache.

**Implementation methodology.** `load_state()` pokazuje stan bez I/O. Cache LRU jest opcjonalny i lokalny; wywołanie `clear_cache` zwalnia tylko referencje cache, nie obiekty nadal przypisane przez użytkownika.

**Figure description.** Tabela stanów przed i po filtrze oraz odczycie; wyniki porównania tożsamości obiektów przy wyłączonym i włączonym cache.

In [ ]:
states = [{"stage": "constructed", **store.load_state()}]
family = store.family(selection["family"])
assert not any(store.load_state().values())
_ = store.family_ids(participant=selection["participant"])
states.append({"stage": "manifest_filter", **store.load_state()})
first = family.alignment
second = family.alignment
states.append({"stage": "two_alignment_reads", **store.load_state()})
assert first is not second
assert store.load_state()["cached_objects"] == 0
cache_store = DataStore(store.root, cache_entries=settings["loading"]["cache_entries"])
cached_first = cache_store.family(family.accession).alignment
assert cache_store.family(family.accession).alignment is cached_first
states.append({"stage": "explicit_cache", **cache_store.load_state()})
cache_store.clear_cache()
assert cache_store.load_state()["cached_objects"] == 0
state_table = pd.DataFrame(states)
display(state_table)
del first, second, cached_first, cache_store

### Remarks

Pamięć parsera nadal obejmuje całe wybrane MSA, pojedynczy plik mmCIF lub tekst jednego raportu. ZIP jest dekompresowany członek po członku; nie jest ładowany jako 14 GB danych. Referencje notebooka i zapisane wyniki zwiększają pamięć niezależnie od cache.

### Notes

## Plan artifact keys before reading

### Methodology

**Theoretical methodology.** Lista rodzin wynika z filtra uczestnika; wspólne PDB są deduplikowane w `iter_requests`. Jawny plan pozwala policzyć i ograniczyć pracę zanim nastąpi odczyt payloadów.

**Implementation methodology.** Dla demonstracji bierzemy kilka rodzin, tylko trzy rodzaje źródeł rodzinnych. Oddzielna jawna lista zawiera sześć typów czytników dla jednej wybranej rodziny i jednego PDB.

**Figure description.** Tabela pokazuje klucze zadań bez załadowanych obiektów.

In [ ]:
from dataclasses import asdict
from aminor_msa.io.parallel import SourceRequest, iter_requests, iter_loaded, summarize_request

family_limit = settings["loading"]["family_limit"]
family_ids = store.family_ids(participant=selection["participant"])[:family_limit]
family_requests = list(iter_requests(store, participant=selection["participant"], families=family_ids,
                                    kinds=("alignment", "metadata", "cm")))
requests = [SourceRequest(kind, selection["family"]) for kind in ("alignment", "metadata", "cm")]
requests += [SourceRequest(kind, selection["pdb_id"]) for kind in ("coordinates", "cif")]
requests += [SourceRequest("dssr", selection["pdb_id"], selection["report_variant"])]
plan = pd.DataFrame([asdict(request) for request in requests])
display(pd.DataFrame([asdict(request) for request in family_requests]))
display(plan)
assert len(family_requests) == 3 * len(family_ids)
assert len(requests) == len(set(requests))

### Remarks

Nie wywołuj `list(iter_loaded(...))` dla całego zbioru z domyślnym zwracaniem obiektów. Strumieniuj wyniki albo przygotuj mały wynik w workerze. Przykład poniżej zwraca tylko skalary.

### Notes

## Bounded thread and process loading

### Methodology

**Theoretical methodology.** Każdy worker ma własny DataStore bez cache i leniwie otwierany ZIP wielokrotnego użytku. Kolejka jest ograniczona przez prefetch; wyniki zachowują kolejność planu.

**Implementation methodology.** `summarize_request` jest importowalną funkcją modułową: czyta plik w workerze i zwraca tylko liczności. `closing` zamyka iterator także przy wcześniejszym przerwaniu. Backend process używa spawn, thread unika kopiowania danych między procesami.

**Figure description.** Dwie tabele porównują odczyty tych samych sześciu źródeł przez wątki i procesy. Nie jest to benchmark przepustowości ani pomiar RSS.

In [ ]:
from contextlib import closing

workers = settings["loading"]["workers"]
prefetch = settings["loading"]["prefetch"]
backend = "thread"
with closing(iter_loaded(store, requests, workers=workers, prefetch=prefetch, backend=backend,
                         transform=summarize_request)) as stream:
    threaded = pd.DataFrame([loaded.data for loaded in stream])
display(threaded)
assert threaded["kind"].tolist() == plan["kind"].tolist()

In [ ]:
workers = settings["loading"]["workers"]
prefetch = settings["loading"]["prefetch"]
backend = "process"
with closing(iter_loaded(store, requests, workers=workers, prefetch=prefetch, backend=backend,
                         transform=summarize_request)) as stream:
    processes = pd.DataFrame([loaded.data for loaded in stream])
display(processes)
pd.testing.assert_frame_equal(threaded, processes)

### Remarks

Własny transform umieść w module `.py`, żeby worker spawn mógł go zaimportować; skrypt uruchamiający procesy wymaga `if __name__ == "__main__"`. Do treningu transform powinien przygotować cechy/shard i zwrócić jego ścieżkę. Tensorów treningowych i shardów jeszcze nie implementujemy; będą w features/training.datasets. Więcej workerów może zwiększyć pamięć i obciążyć dysk. Zacznij od 2/2 i mierz własny pipeline.

### Notes

## Results / Summary

### LLM

Przykład sprawdza domyślny brak cache, jawne ponowne użycie obiektu z LRU oraz identyczność podsumowań sześciu źródeł z wątków i procesów. Kolejka pozostaje ograniczona przez prefetch; limit nie obejmuje pamięci obiektów, które aplikacja sama zachowuje. Gotowy szybki format treningowy wymaga osobnego etapu przygotowania cech.

### Person